# Exploratory analysis — production quality and defect risk

This notebook analyses a **synthetic** manufacturing dataset created for a portfolio demonstration. It answers ten business questions, separates production volume from defect rate, and concludes with an accessible statistical comparison. Findings indicate associations in simulated data; they do not prove causation or establish readiness for a real factory.

## Setup and data validation

The processed dataset is used because invalid values, duplicates, and missing values have already been handled by the reproducible cleaning pipeline. The checks below protect the analysis from silently using an unexpected schema.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency, levene, shapiro, ttest_ind

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "production_quality_clean.csv"

data = pd.read_csv(DATA_PATH, parse_dates=["production_date"])
required = {"production_id", "production_date", "machine_id", "site", "material_type", "operator_team", "temperature", "pressure", "production_duration", "quality_score", "production_shift", "defect"}
assert required.issubset(data.columns)
assert data["production_id"].is_unique
assert data.isna().sum().sum() == 0
print(f"Loaded {len(data):,} clean production records with {data.shape[1]} columns.")

The checks confirm that every production identifier is unique and that the analysis contains no missing values. Because the source is synthetic, exact rates reflect the simulation design and should not be treated as an external benchmark.

## 1. What is the overall defect rate?

Both the number and the rate matter: counts describe workload, while the rate makes performance comparable across samples of different sizes.

In [ ]:
defect_summary = data["defect"].value_counts().rename(index={0: "Acceptable", 1: "Defective"}).reset_index()
defect_summary.columns = ["status", "count"]
overall_rate = data["defect"].mean()
print(f"Overall defect rate: {overall_rate:.2%} ({data['defect'].sum():,} of {len(data):,} products)")

fig = px.bar(defect_summary, x="status", y="count", color="status", title="Product count by quality status", labels={"status": "Quality status", "count": "Number of products"}, text="count", color_discrete_map={"Acceptable": "#3B7A57", "Defective": "#B44C43"})
fig.update_layout(showlegend=False)
fig.show()

**Interpretation.** 208 of 2,500 records are defective, an overall rate of 8.32%. **Business implication.** This baseline can support consistent site and machine comparisons. **Caution.** The rate is deliberately simulated and does not represent an industrial benchmark.

## 2. Which machine has the highest defect rate?

The paired view prevents a low-volume machine from appearing important based only on a rate or a high-volume machine from appearing poor based only on its count.

In [ ]:
machine_quality = data.groupby("machine_id")["defect"].agg(defect_count="sum", production_volume="size", defect_rate="mean").reset_index().sort_values("defect_rate", ascending=False)
display(machine_quality.style.format({"defect_rate": "{:.2%}"}))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Defect count and production volume", "Defect rate"))
fig.add_trace(go.Bar(x=machine_quality["machine_id"], y=machine_quality["production_volume"], name="Production volume", marker_color="#9FB3C8"), row=1, col=1)
fig.add_trace(go.Bar(x=machine_quality["machine_id"], y=machine_quality["defect_count"], name="Defect count", marker_color="#B44C43"), row=1, col=1)
fig.add_trace(go.Bar(x=machine_quality["machine_id"], y=machine_quality["defect_rate"], name="Defect rate", marker_color="#C98A3D", text=machine_quality["defect_rate"].map(lambda value: f"{value:.1%}")), row=1, col=2)
fig.update_yaxes(title_text="Number of products", row=1, col=1)
fig.update_yaxes(title_text="Defect rate", tickformat=".0%", row=1, col=2)
fig.update_layout(title="Machine quality performance: counts, volume, and rates", barmode="group")
fig.show()

**Interpretation.** M-07 has the highest observed rate (12.16%, 40 defects from 329 products), followed by M-06 (10.80%). **Business implication.** M-07 is the strongest candidate for a maintenance and parameter review. **Caution.** Machine mix, material mix, and random variation may partly explain differences; the chart does not establish a machine fault.

## 3. Which site has the highest defect rate?

In [ ]:
site_quality = data.groupby("site")["defect"].agg(defect_count="sum", production_volume="size", defect_rate="mean").reset_index().sort_values("defect_rate", ascending=False)
fig = px.bar(site_quality, x="site", y="defect_rate", title="Defect rate by production site", labels={"site": "Site", "defect_rate": "Defect rate"}, text=site_quality.apply(lambda row: f"{row.defect_rate:.1%}<br>n={row.production_volume:,}", axis=1), color="defect_rate", color_continuous_scale="Oranges")
fig.update_yaxes(tickformat=".0%")
fig.show()

**Interpretation.** Besancon has the highest observed site rate (9.49%, 61 defects from 643 products), while Lyon has the largest volume. **Business implication.** Compare Besancon's equipment and process mix with the other sites. **Caution.** Site effects are confounded with machine allocation because each simulated machine belongs to one site.

## 4. How does the defect rate evolve over time?

Monthly aggregation gives more stable rates than individual days while preserving a useful operational trend.

In [ ]:
monthly = data.set_index("production_date").resample("MS")["defect"].agg(defect_count="sum", production_volume="size", defect_rate="mean").reset_index()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=monthly["production_date"], y=monthly["production_volume"], name="Production volume", marker_color="#C8D5E3", opacity=0.55), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly["production_date"], y=monthly["defect_rate"], name="Defect rate", mode="lines+markers", line=dict(color="#B44C43", width=2)), secondary_y=True)
fig.update_yaxes(title_text="Production volume", secondary_y=False)
fig.update_yaxes(title_text="Defect rate", tickformat=".0%", secondary_y=True)
fig.update_layout(title="Monthly production volume and defect rate", xaxis_title="Production month")
fig.show()

**Interpretation.** Monthly rates fluctuate around the overall baseline without a persistent trend. **Business implication.** Quality teams should investigate individual peaks alongside maintenance or material records. **Caution.** Some variation is sampling noise, and the synthetic dataset has no real interventions or seasonality labels.

## 5. Is temperature associated with defects?

The histogram shows operating coverage; the box plot compares the two outcome groups without implying causality.

In [ ]:
fig = px.histogram(data, x="temperature", color=data["defect"].map({0: "Acceptable", 1: "Defective"}), nbins=45, barmode="overlay", opacity=0.65, title="Machine temperature distribution by quality status", labels={"temperature": "Temperature (°C)", "color": "Quality status", "count": "Production records"}, color_discrete_map={"Acceptable": "#3B7A57", "Defective": "#B44C43"})
fig.show()

plot_data = data.assign(quality_status=data["defect"].map({0: "Acceptable", 1: "Defective"}))
fig = px.box(plot_data, x="quality_status", y="temperature", color="quality_status", points="outliers", title="Temperature by defect status", labels={"quality_status": "Quality status", "temperature": "Temperature (°C)"}, color_discrete_map={"Acceptable": "#3B7A57", "Defective": "#B44C43"})
fig.update_layout(showlegend=False)
fig.show()

**Interpretation.** Defective products have a slightly higher mean temperature (70.64°C versus 70.35°C) and wider spread, but the distributions overlap strongly. **Business implication.** Monitoring distance from the normal setting is more useful than treating temperature alone as a simple threshold. **Caution.** The visual difference is small; a formal test appears later and correlation does not establish causation.

## 6. Is pressure associated with defects?

In [ ]:
fig = px.box(plot_data, x="quality_status", y="pressure", color="quality_status", points="outliers", title="Production pressure by defect status", labels={"quality_status": "Quality status", "pressure": "Pressure (bar)"}, color_discrete_map={"Acceptable": "#3B7A57", "Defective": "#B44C43"})
fig.update_layout(showlegend=False)
fig.show()

**Interpretation.** Defective products show a somewhat higher mean pressure and greater variability (5.14 versus 5.00 bar). **Business implication.** Review production events far from the 5-bar operating target. **Caution.** The box plot is unadjusted for machine, material, and temperature, so it cannot isolate a pressure effect.

## 7. Does production duration affect quality or defect risk?

In [ ]:
fig = px.box(plot_data, x="quality_status", y="production_duration", color="quality_status", points="outliers", title="Production duration by defect status", labels={"quality_status": "Quality status", "production_duration": "Duration (minutes)"}, color_discrete_map={"Acceptable": "#3B7A57", "Defective": "#B44C43"})
fig.update_layout(showlegend=False)
fig.show()

duration_quality_correlation = data[["production_duration", "quality_score"]].corr().iloc[0, 1]
print(f"Correlation between duration and quality score: {duration_quality_correlation:.3f}")

**Interpretation.** Defective products have a slightly longer mean duration (45.42 versus 44.91 minutes), with substantial overlap and a few long runs. **Business implication.** Investigate unusually long cycles as process signals, especially in combination with other deviations. **Caution.** Material type affects planned duration, so longer production is not automatically poor performance.

## 8. Which material type has the highest defect rate?

In [ ]:
material_quality = data.groupby("material_type")["defect"].agg(defect_count="sum", production_volume="size", defect_rate="mean").reset_index().sort_values("defect_rate", ascending=False)
fig = px.bar(material_quality, x="material_type", y="defect_rate", color="defect_rate", title="Defect rate by material type", labels={"material_type": "Material type", "defect_rate": "Defect rate"}, text=material_quality.apply(lambda row: f"{row.defect_rate:.1%}<br>n={row.production_volume:,}", axis=1), color_continuous_scale="Oranges")
fig.update_yaxes(tickformat=".0%")
fig.show()

**Interpretation.** Ceramic has the highest observed rate (10.40%, 57 defects from 548 products). **Business implication.** Review ceramic-specific temperature, pressure, and handling windows with process experts. **Caution.** The result is an association in simulated data and may reflect the deliberately encoded material sensitivity.

## 9. Are some operator teams associated with more defects?

In [ ]:
team_quality = data.groupby("operator_team")["defect"].agg(defect_count="sum", production_volume="size", defect_rate="mean").reset_index().sort_values("defect_rate", ascending=False)
fig = px.bar(team_quality, x="operator_team", y="defect_rate", color="defect_rate", title="Defect rate by operator team", labels={"operator_team": "Operator team", "defect_rate": "Defect rate"}, text=team_quality.apply(lambda row: f"{row.defect_rate:.1%}<br>n={row.production_volume:,}", axis=1), color_continuous_scale="Oranges")
fig.update_yaxes(tickformat=".0%")
fig.show()

**Interpretation.** Team C has the highest observed rate (9.23%), but team rates are relatively close. **Business implication.** Treat this as a prompt to compare assignments and operating contexts, not as a performance ranking. **Caution.** Team comparisons may reflect machine, shift, or material allocation and should never be used for individual blame.

## 10. During which shift are defects most frequent?

“Most frequent” is reported as both count and rate so the answer does not depend only on shift volume.

In [ ]:
shift_quality = data.groupby("production_shift")["defect"].agg(defect_count="sum", production_volume="size", defect_rate="mean").reset_index().sort_values("defect_rate", ascending=False)
fig = px.bar(shift_quality, x="production_shift", y="defect_rate", color="defect_rate", title="Defect rate by production shift", labels={"production_shift": "Production shift", "defect_rate": "Defect rate"}, text=shift_quality.apply(lambda row: f"{row.defect_rate:.1%}<br>{row.defect_count:.0f} defects / n={row.production_volume:,}", axis=1), color_continuous_scale="Oranges")
fig.update_yaxes(tickformat=".0%")
fig.show()

**Interpretation.** Afternoon has both the highest observed rate (9.40%) and count (82 defects). **Business implication.** Compare afternoon operating conditions and product mix before considering interventions. **Caution.** Shift is a broad time category and is not evidence about staffing quality or fatigue.

## Correlation overview

The matrix summarizes linear relationships between numerical variables. `quality_score` is included only for retrospective analysis and must be excluded from defect prediction because it is measured during or after inspection.

In [ ]:
numeric_columns = ["temperature", "pressure", "production_duration", "quality_score", "temperature_deviation", "pressure_deviation", "defect"]
correlations = data[numeric_columns].corr()
fig = px.imshow(correlations, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Correlation matrix for numerical quality variables", labels={"color": "Pearson correlation"})
fig.show()

**Interpretation.** Quality score is strongly negatively related to defects by construction, while individual operational variables show weaker linear associations. **Business implication.** Multivariable modelling can combine several small operational signals. **Caution.** Quality score is leakage for prediction, correlations miss non-linear patterns, and none of these relationships proves causation.

## Statistical analysis: is average temperature different by defect status?

**Null hypothesis (H₀):** acceptable and defective products have the same average machine temperature.  
**Alternative hypothesis (H₁):** their average machine temperatures differ.

We first inspect group sizes, distributions, and variability. Because the groups are independent and large but their variances differ, Welch's independent-samples t-test is preferable to the equal-variance t-test. Welch's test directly addresses the difference in group means and is robust to unequal sample sizes and variances.

In [ ]:
temperature_summary = data.groupby("defect")["temperature"].agg(["count", "mean", "median", "std", "min", "max"])
temperature_summary.index = temperature_summary.index.map({0: "Acceptable", 1: "Defective"})
display(temperature_summary)

acceptable_temperature = data.loc[data["defect"].eq(0), "temperature"]
defective_temperature = data.loc[data["defect"].eq(1), "temperature"]

shapiro_defective = shapiro(defective_temperature)
shapiro_acceptable_sample = shapiro(acceptable_temperature.sample(500, random_state=42))
variance_test = levene(defective_temperature, acceptable_temperature, center="median")
print(f"Shapiro p-value, defective group: {shapiro_defective.pvalue:.4f}")
print(f"Shapiro p-value, 500-record acceptable sample: {shapiro_acceptable_sample.pvalue:.4g}")
print(f"Brown–Forsythe variance-test p-value: {variance_test.pvalue:.4g}")

The defective group contains 208 records and the acceptable group 2,292. Histograms and box plots show strong overlap. A normality test on a large acceptable sample is sensitive enough to reject exact normality, and the variance test indicates unequal spread. With these large independent groups, Welch's test remains an appropriate, accessible comparison of means; the unequal-variance version avoids the standard t-test's equal-variance assumption.

In [ ]:
t_statistic, p_value = ttest_ind(defective_temperature, acceptable_temperature, equal_var=False)
n_defective, n_acceptable = len(defective_temperature), len(acceptable_temperature)
pooled_sd = np.sqrt(((n_defective - 1) * defective_temperature.var(ddof=1) + (n_acceptable - 1) * acceptable_temperature.var(ddof=1)) / (n_defective + n_acceptable - 2))
cohens_d = (defective_temperature.mean() - acceptable_temperature.mean()) / pooled_sd

print(f"Welch t-statistic: {t_statistic:.3f}")
print(f"Two-sided p-value: {p_value:.3f}")
print(f"Cohen's d: {cohens_d:.3f}")

At a 5% significance level, **p = 0.520**, so the analysis does not reject the null hypothesis. The estimated mean difference is only 0.29°C, and Cohen's *d* = 0.054 indicates a negligible standardized effect. For a manager, this means average temperature alone does not clearly separate defective and acceptable products in this sample. Temperature deviations may still matter jointly with pressure, material, or machine.

Statistical significance is not proof of causation, and a non-significant result is not proof of no operational relationship. Limitations include synthetic data, unequal group sizes, deliberately introduced outliers, unmeasured interactions, repeated machine contexts, and the fact that this test does not adjust for other variables.

### Optional machine–defect association check

A chi-square test checks whether defect status and machine assignment are statistically independent. It is exploratory and does not identify a causal machine effect.

In [ ]:
machine_table = pd.crosstab(data["machine_id"], data["defect"])
chi2_statistic, chi2_p_value, degrees_freedom, expected = chi2_contingency(machine_table)
print(f"Chi-square statistic: {chi2_statistic:.3f}")
print(f"Degrees of freedom: {degrees_freedom}")
print(f"p-value: {chi2_p_value:.3f}")
print(f"Minimum expected cell count: {expected.min():.2f}")

The machine association test gives **p = 0.099**, so it does not reach the conventional 5% threshold despite M-07's higher observed rate. Operationally, M-07 remains worth monitoring, but the evidence does not justify claiming a proven machine effect. Larger real samples and adjusted modelling would be needed.

## Business conclusions and next steps

- Prioritize investigation of M-07, ceramic production, and large pressure or temperature deviations, while accounting for volume and product mix.
- Use the dashboard to monitor patterns rather than assign blame to teams or sites.
- Validate all findings with production experts and physical inspection records.
- Collect real process, maintenance, batch, supplier, and inspection data before considering deployment.
- Treat every result here as a synthetic demonstration: correlation and model association do not establish causation.